In [2]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt    
import seaborn as sns  
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer 


In [3]:
df = pd.read_csv('titanic_data_updated (1).csv')
df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,no,third,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,yes,first,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,yes,third,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,yes,first,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,no,third,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,no,second,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,yes,first,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,no,third,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,yes,first,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


In [7]:
df.drop(['PassengerId','Name','Ticket'],axis = 1, inplace =True)
x = df.drop('Survived',axis = 1)
y = df['Survived']


In [8]:
df

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Cabin,Embarked
0,no,third,male,22.0,1,0,7.2500,NaN,S
1,yes,first,female,38.0,1,0,71.2833,C85,C
2,yes,third,female,26.0,0,0,7.9250,NaN,S
3,yes,first,female,35.0,1,0,53.1000,C123,S
4,no,third,male,35.0,0,0,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...
886,no,second,male,27.0,0,0,13.0000,NaN,S
887,yes,first,female,19.0,0,0,30.0000,B42,S
888,no,third,female,NaN,1,2,23.4500,NaN,S
889,yes,first,male,26.0,0,0,30.0000,C148,C


In [9]:
# Split the dataset into training and testing sets
# Dataset ke train (80%) ebong test (20%) data te vag kora hocche
X_train, X_test, y_train, y_test = train_test_split(
    x, y,
    test_size=0.2,
    random_state=42
)

# ==========================================================
# AGE COLUMN IMPUTATION
# Age column er missing values mean (average) diye fill kora hobe
# ==========================================================

# Create an imputer that replaces NaN values with the mean value
# NaN values gulo Age column er average value diye replace korbe
age_imputer = SimpleImputer(
    missing_values=np.nan,
    strategy='mean'
)

# Learn the mean age from training data only
# Training data theke Age column er average ber kora hocche
age_imputer.fit(X_train[['Age']])

# Replace missing Age values in training data
# Train dataset er missing Age values average diye fill kora hocche
X_train['Age'] = age_imputer.transform(
    X_train[['Age']]
).ravel()

# Replace missing Age values in test data using the same mean
# Test dataset eo train data theke paoa same average use kora hocche
X_test['Age'] = age_imputer.transform(
    X_test[['Age']]
)

# Check if any missing values remain in test dataset
# Test data te ekhono kono missing value ache kina check kora hocche
X_test.isnull().sum()


# ==========================================================
# EMBARKED COLUMN IMPUTATION
# Embarked column er missing values most frequent value diye fill kora hobe
# ==========================================================

# Create an imputer using the most frequent category
# Sobcheye beshi bar thaka category diye missing values fill korbe
embarked_imputer = SimpleImputer(
    missing_values=np.nan,
    strategy='most_frequent'
)

# Learn the most common Embarked value from training data
# Training data theke sobcheye common Embarked value ber kora hocche
embarked_imputer.fit(X_train[['Embarked']])

# Replace missing values in training data
# Train data er missing Embarked values fill kora hocche
X_train['Embarked'] = embarked_imputer.transform(
    X_train[['Embarked']]
).ravel()

# Replace missing values in test data using same learned value
# Test data teo same most frequent value use kora hocche
X_test['Embarked'] = embarked_imputer.transform(
    X_test[['Embarked']]
).ravel()


# ==========================================================
# CABIN COLUMN IMPUTATION
# Cabin column er missing values "Missing" text diye fill kora hobe
# Sathe ekta indicator column add hobe
# ==========================================================

# Create an imputer that:
# 1. Replaces missing Cabin values with "Missing"
# 2. Adds a new indicator column showing whether value was missing
#
# Ei imputer:
# 1. Missing Cabin values ke "Missing" diye replace korbe
# 2. Ekta extra column add korbe jekhane missing chilo kina dekhabe
cabin_imputer = SimpleImputer(
    missing_values=np.nan,
    strategy='constant',
    fill_value='Missing',
    add_indicator=True
)

# Learn from training data
# Training data diye imputer fit kora hocche
cabin_imputer.fit(X_train[['Cabin']])

# Transform training data
# Cabin value fill kora hocche ebong indicator column add hocche
X_train[['Cabin', 'cabin_missing_indicator']] = (
    cabin_imputer.transform(X_train[['Cabin']])
)

# Transform test data in the same way
# Test data teo same rule apply kora hocche
X_test[['Cabin', 'cabin_missing_indicator']] = (
    cabin_imputer.transform(X_test[['Cabin']])
)

### Z_Score Outliers 

In [10]:
 
mean_of_age = X_train['Age'].mean()
std_of_age  = X_train['Age'].std()

X_train['z_score_Age'] = (X_train['Age'] - mean_of_age) / std_of_age

X_train


,Pclass,Sex,Age,SibSp,Parch,Fare,Cabin,Embarked,cabin_missing_indicator,z_score_Age
331,first,male,45.500000,0,0,28.5000,C124,S,False,1.231398e+00
733,second,male,23.000000,0,0,13.0000,Missing,S,True,-5.001304e-01
382,third,male,32.000000,0,0,7.9250,Missing,S,True,1.924808e-01
704,third,male,26.000000,1,0,7.8542,Missing,S,True,-2.692600e-01
813,third,female,6.000000,4,2,31.2750,Missing,S,True,-1.808396e+00
...,...,...,...,...,...,...,...,...,...,...
106,third,female,21.000000,0,0,7.6500,Missing,S,True,-6.540440e-01
270,first,male,29.498846,0,0,31.0000,Missing,S,True,2.734055e-16
860,third,male,41.000000,2,0,14.1083,Missing,S,True,8.850920e-01
435,first,female,14.000000,1,2,120.0000,B96 B98,S,False,-1.192742e+00


In [11]:
# for age fine the amount the outliers

outliers_of_age = X_train[abs(X_train['z_score_Age']) >3]
print(len(outliers_of_age))
display(outliers_of_age)

5


,Pclass,Sex,Age,SibSp,Parch,Fare,Cabin,Embarked,cabin_missing_indicator,z_score_Age
116,third,male,70.5,0,0,7.750,Missing,Q,True,3.155317
745,first,male,70.0,1,1,71.000,B22,S,False,3.116839
630,first,male,80.0,0,0,30.000,A23,S,False,3.886407
851,third,male,74.0,0,0,7.775,Missing,S,True,3.424666
672,second,male,70.0,0,0,10.500,Missing,S,True,3.116839


In [12]:
mean_of_fare= X_train['Fare'].mean()
std_of_fare  = X_train['Fare'].std()

X_train['z_score_Fare'] = (X_train['Fare'] - mean_of_fare) / std_of_fare

X_train


,Pclass,Sex,Age,SibSp,Parch,Fare,Cabin,Embarked,cabin_missing_indicator,z_score_Age,z_score_Fare
331,first,male,45.500000,0,0,28.5000,C124,S,False,1.231398e+00,-0.078628
733,second,male,23.000000,0,0,13.0000,Missing,S,True,-5.001304e-01,-0.376880
382,third,male,32.000000,0,0,7.9250,Missing,S,True,1.924808e-01,-0.474533
704,third,male,26.000000,1,0,7.8542,Missing,S,True,-2.692600e-01,-0.475896
813,third,female,6.000000,4,2,31.2750,Missing,S,True,-1.808396e+00,-0.025232
...,...,...,...,...,...,...,...,...,...,...,...
106,third,female,21.000000,0,0,7.6500,Missing,S,True,-6.540440e-01,-0.479825
270,first,male,29.498846,0,0,31.0000,Missing,S,True,2.734055e-16,-0.030523
860,third,male,41.000000,2,0,14.1083,Missing,S,True,8.850920e-01,-0.355554
435,first,female,14.000000,1,2,120.0000,B96 B98,S,False,-1.192742e+00,1.682019


In [13]:
outliers_of_fare = X_train[abs(X_train['z_score_Fare']) >3]
print(len(outliers_of_fare))
display(outliers_of_fare)

17


,Pclass,Sex,Age,SibSp,Parch,Fare,Cabin,Embarked,cabin_missing_indicator,z_score_Age,z_score_Fare
118,first,male,24.000000,0,1,247.5208,B58 B60,C,False,-4.231736e-01,4.135780
716,first,female,38.000000,0,0,227.5250,C45,C,False,6.542216e-01,3.751020
377,first,male,27.000000,0,2,211.5000,C82,C,False,-1.923032e-01,3.442666
742,first,female,21.000000,2,2,262.3750,B57 B59 B63 B66,C,False,-6.540440e-01,4.421605
380,first,female,42.000000,0,0,227.5250,Missing,C,True,9.620488e-01,3.751020
779,first,female,43.000000,0,1,211.3375,B3,S,False,1.039006e+00,3.439539
730,first,female,29.000000,0,0,211.3375,B5,S,False,-3.838960e-02,3.439539
88,first,female,23.000000,3,2,263.0000,C23 C25 C27,S,False,-5.001304e-01,4.433631
341,first,female,24.000000,3,2,263.0000,C23 C25 C27,S,False,-4.231736e-01,4.433631
557,first,male,29.498846,0,0,227.5250,Missing,C,True,2.734055e-16,3.751020


### Outliers with IQR

In [14]:
# age 
q1= X_train['Age'].quantile(0.25)
q3 = X_train['Age'].quantile(0.75)

IQR = q3 - q1
age_minimum = q1 - 1.5 * IQR
age_maximum = q3 + 1.5 * IQR

print(age_minimum)
print(age_maximum)


2.5
54.5


In [15]:
# for age outliers by IQR
age_outliers_IQR = X_train[(X_train['Age']<age_minimum) | (X_train['Age'] > age_maximum)]


print(len(age_outliers_IQR))
display(age_outliers_IQR)

54


,Pclass,Sex,Age,SibSp,Parch,Fare,Cabin,Embarked,cabin_missing_indicator,z_score_Age,z_score_Fare
326,third,male,61.00,0,0,6.2375,Missing,S,True,2.424228,-0.507004
483,third,female,63.00,0,0,9.5875,Missing,S,True,2.578141,-0.442543
7,third,male,2.00,3,1,21.0750,Missing,S,True,-2.116223,-0.221500
305,first,male,0.92,1,2,151.5500,C22 C26,S,False,-2.199336,2.289105
829,first,female,62.00,0,0,80.0000,B28,S,False,2.501185,0.912337
626,second,male,57.00,0,0,12.3500,Missing,Q,True,2.116401,-0.389387
456,first,male,65.00,0,0,26.5500,E38,S,False,2.732055,-0.116150
172,third,female,1.00,1,1,11.1333,Missing,S,True,-2.193180,-0.412799
164,third,male,1.00,4,1,39.6875,Missing,S,True,-2.193180,0.136642
381,third,female,1.00,0,2,15.7417,Missing,C,True,-2.193180,-0.324124


In [16]:
# Fare 
q1= X_train['Fare'].quantile(0.25)
q3 = X_train['Fare'].quantile(0.75)

fare_IQR = q3 - q1
fare_minimum = q1 - 1.5 * fare_IQR
fare_maximum = q3 + 1.5 * fare_IQR

print(fare_minimum)
print(fare_maximum)

-25.937499999999996
64.3625


In [17]:
fare_outliers_IQR = X_train[(X_train['Fare'] < fare_minimum) | (X_train['Fare'] > fare_maximum)]


print(len(fare_outliers_IQR))
display(fare_outliers_IQR)

96


,Pclass,Sex,Age,SibSp,Parch,Fare,Cabin,Embarked,cabin_missing_indicator,z_score_Age,z_score_Fare
118,first,male,24.0,0,1,247.5208,B58 B60,C,False,-0.423174,4.135780
486,first,female,35.0,1,0,90.0000,C93,S,False,0.423351,1.104757
716,first,female,38.0,0,0,227.5250,C45,C,False,0.654222,3.751020
390,first,male,36.0,1,2,120.0000,B96 B98,S,False,0.500308,1.682019
377,first,male,27.0,0,2,211.5000,C82,C,False,-0.192303,3.442666
...,...,...,...,...,...,...,...,...,...,...,...
681,first,male,27.0,0,0,76.7292,D49,C,False,-0.192303,0.849400
385,second,male,18.0,0,0,73.5000,Missing,S,True,-0.884914,0.787264
700,first,female,18.0,1,0,227.5250,C62 C64,C,False,-0.884914,3.751020
435,first,female,14.0,1,2,120.0000,B96 B98,S,False,-1.192742,1.682019


In [19]:
# z score outliers ditection is better for  finding age outliers
# so we delete outliers  from X_train data 
# in case those value we count when value <= 3

X_train = X_train[abs(X_train['z_score_Age']) <= 3]
X_train


,Pclass,Sex,Age,SibSp,Parch,Fare,Cabin,Embarked,cabin_missing_indicator,z_score_Age,z_score_Fare
331,first,male,45.500000,0,0,28.5000,C124,S,False,1.231398e+00,-0.078628
733,second,male,23.000000,0,0,13.0000,Missing,S,True,-5.001304e-01,-0.376880
382,third,male,32.000000,0,0,7.9250,Missing,S,True,1.924808e-01,-0.474533
704,third,male,26.000000,1,0,7.8542,Missing,S,True,-2.692600e-01,-0.475896
813,third,female,6.000000,4,2,31.2750,Missing,S,True,-1.808396e+00,-0.025232
...,...,...,...,...,...,...,...,...,...,...,...
106,third,female,21.000000,0,0,7.6500,Missing,S,True,-6.540440e-01,-0.479825
270,first,male,29.498846,0,0,31.0000,Missing,S,True,2.734055e-16,-0.030523
860,third,male,41.000000,2,0,14.1083,Missing,S,True,8.850920e-01,-0.355554
435,first,female,14.000000,1,2,120.0000,B96 B98,S,False,-1.192742e+00,1.682019


In [22]:
minimum_zscore_fare = X_train[abs(X_train['z_score_Fare']) <= 3 ]['Fare'].max()

print(minimum_zscore_fare)
 

164.8667


In [25]:
df.describe()

,Age,SibSp,Parch,Fare
count,714.000000,891.000000,891.000000,891.000000
mean,29.699118,0.523008,0.381594,32.204208
std,14.526497,1.102743,0.806057,49.693429
min,0.420000,0.000000,0.000000,0.000000
25%,20.125000,0.000000,0.000000,7.910400
50%,28.000000,0.000000,0.000000,14.454200
75%,38.000000,1.000000,0.000000,31.000000
max,80.000000,8.000000,6.000000,512.329200


In [20]:
# Calculate IQR (Interquartile Range) for the Fare column
# Fare column er outlier detect korar jonno IQR method use kora hocche

q1= X_train['Fare'].quantile(0.25)
q3 = X_train['Fare'].quantile(0.75)

fare_IQR = q3 - q1
fare_minimum = max(0,q1 - 1.5 * fare_IQR)
fare_maximum = q3 + 1.5 * fare_IQR

print(fare_minimum)
print(fare_maximum)




0
64.3625


In [27]:

# Clip (cap) the Fare values within the specified range

# fare_minimum er niche thaka value gulo fare_minimum hoye jabe
# fare_maximum er upore thaka value gulo fare_maximum hoye jabe
# Majher value gulo unchanged thakbe
# Ei process ke outlier capping ba winsorization bola hoy

X_train['Fare'] = X_train['Fare'].clip(fare_minimum, fare_maximum)

# Display the maximum Fare value after capping

# Clip korar por Fare column er highest value dekha hocche
# Eta normally fare_maximum er soman hobe
X_train['Fare'].max()

C:\Users\ASUS\AppData\Local\Temp\ipykernel_14708\1103440507.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train['Fare'] = X_train['Fare'].clip(fare_minimum,fare_maximum)


64.3625